# Multi-GPU Latent Communication (Token-to-Token Translation)

This notebook implements **1-to-1 Token Latent Translation**. 
By matching the exact sequence length of Model A and Model B, the Aligner only needs to learn "how" to translate latent concepts linearly, rather than learning "where" to route them with attention. 
It uses HuggingFace's `device_map="auto"` to seamlessly distribute the models across the T4 GPUs.

In [ ]:
!pip install torch transformers datasets accelerate bitsandbytes

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import gc

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

### 1. Token-to-Token Architecture
A Deep Residual MLP applied independently to every token in the sequence.

In [ ]:
class TokenToTokenAligner(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.proj = nn.Linear(input_dim, output_dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(output_dim, output_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(output_dim * 4, output_dim)
        )
        self.norm1 = nn.LayerNorm(output_dim)
        self.norm2 = nn.LayerNorm(output_dim)

    def forward(self, hidden_a):
        # hidden_a: [batch, seq_len, input_dim]
        x = self.proj(hidden_a)
        x = self.norm1(x)
        
        # Residual connection
        ffn_out = self.ffn(x)
        out = self.norm2(x + ffn_out)
        
        return out # [batch, seq_len, output_dim]


### 2. Loading Models with `device_map="auto"`

In [ ]:
MODEL_A = "Qwen/Qwen2.5-0.5B"
MODEL_B = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading Model A...")
tokenizer_a = AutoTokenizer.from_pretrained(MODEL_A)
model_a = AutoModelForCausalLM.from_pretrained(MODEL_A, torch_dtype=torch.bfloat16, device_map="auto")
model_a.eval()
for param in model_a.parameters():
    param.requires_grad = False

print("Loading Model B...")
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForCausalLM.from_pretrained(MODEL_B, torch_dtype=torch.bfloat16, device_map="auto")
model_b.eval()
for param in model_b.parameters():
    param.requires_grad = False

if tokenizer_a.pad_token is None:
    tokenizer_a.pad_token = tokenizer_a.eos_token
if tokenizer_b.pad_token is None:
    tokenizer_b.pad_token = tokenizer_b.eos_token
model_b.config.pad_token_id = tokenizer_b.pad_token_id

print("Loading Dataset...")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
texts = [t["text"].strip() for t in dataset if len(t["text"].strip()) > 50][:20000]


### 3. Pipeline-Parallel Training Loop

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 3
LEARNING_RATE = 3e-4

aligner = TokenToTokenAligner(
    input_dim=model_a.config.hidden_size, 
    output_dim=model_b.config.hidden_size
).to("cuda")

optimizer = torch.optim.AdamW(aligner.parameters(), lr=LEARNING_RATE)
dataloader = DataLoader(texts, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    aligner.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    optimizer.zero_grad()
    for step, batch_texts in enumerate(progress_bar):
        inputs_a = tokenizer_a(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(model_a.device)
        inputs_b = tokenizer_b(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(model_b.device)
        
        # 1. Model A extracts thoughts
        with torch.no_grad():
            outputs_a = model_a(**inputs_a, output_hidden_states=True)
            hidden_a = outputs_a.hidden_states[-1]
            
        # 2. Aligner translates thoughts 1-to-1
        hidden_a = hidden_a.to("cuda").to(aligner.proj.weight.dtype)
        soft_prompts = aligner(hidden_a) # [batch, seq_len_a, dim_b]
        
        # 3. Model B decoding
        with torch.no_grad():
            embeds_b = model_b.get_input_embeddings()(inputs_b['input_ids'])
            
        soft_prompts = soft_prompts.to(model_b.device).to(embeds_b.dtype)
        
        # Combine the soft prompts with the text embeddings
        full_embeds = torch.cat([soft_prompts, embeds_b], dim=1)
        
        # The prompt mask is exactly the same length as Model A's input mask!
        prompt_mask = inputs_a['attention_mask'].to(model_b.device)
        full_mask = torch.cat([prompt_mask, inputs_b['attention_mask']], dim=1)
        
        outputs_b = model_b(inputs_embeds=full_embeds, attention_mask=full_mask)
        logits = outputs_b.logits
        
        # Cross Entropy Loss
        # We shift logits by the exact sequence length of Model A's input to predict the text
        seq_len_a = soft_prompts.size(1)
        shift_logits = logits[:, seq_len_a-1:-1, :].contiguous()
        shift_labels = inputs_b['input_ids'].contiguous()
        
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=model_b.config.pad_token_id
        )
        
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        progress_bar.set_postfix({"loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})
        
    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(dataloader):.4f}")

torch.save(aligner.state_dict(), "token_aligner.pth")
print("Training complete! Saved to token_aligner.pth")


### 4. Inference / Test Communication

In [ ]:
def test_latent_communication(test_sentence):
    print(f"\nInput to Model A: '{test_sentence}'")
    
    aligner.eval()
    
    with torch.no_grad():
        inputs_a = tokenizer_a([test_sentence], return_tensors="pt", padding=True, truncation=True, max_length=128).to(model_a.device)
        outputs_a = model_a(**inputs_a, output_hidden_states=True)
        hidden_a = outputs_a.hidden_states[-1]
        
        hidden_a = hidden_a.to("cuda").to(aligner.proj.weight.dtype)
        soft_prompts = aligner(hidden_a)
        soft_prompts = soft_prompts.to(model_b.device).to(model_b.dtype)
        
        print("-" * 40)
        print("Model B decoding Soft Prompts...")
        
        generated_ids = []
        current_embeds = soft_prompts
        
        for _ in range(30):
            # Since Model B's cache is not being passed correctly for inputs_embeds in some transformers versions,
            # we just run the full sequence through. It's slower but robust for demonstration.
            outputs = model_b(inputs_embeds=current_embeds)
            next_token_logits = outputs.logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1)
            
            generated_ids.append(next_token_id.item())
            if next_token_id.item() == tokenizer_b.eos_token_id:
                break
                
            new_embed = model_b.get_input_embeddings()(next_token_id.unsqueeze(0))
            current_embeds = torch.cat([current_embeds, new_embed], dim=1)
            
        print("Output:", tokenizer_b.decode(generated_ids, skip_special_tokens=True))
        print("-" * 40)

test_latent_communication("The Apollo 11 mission was the first time humans landed on the moon.")
test_latent_communication("Photosynthesis is the process used by plants to convert light into energy.")
